## Project Structure
```
2DIP_exercise/
│-- data/             # Contains images & videos
│   │-- input/        # 1 image and 1 video for each phase respectively
│   │-- output/       # All output images/videos must be stored here
│-- notebooks/        # Jupyter Notebooks for each phase
│   │-- part1.ipynb   # Image processing & feature extraction
│   │-- part2.ipynb   # Optical flow, object detection and tracking 
│-- README.md         # Project instructions
```

In [1]:
# imports
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# define paths
base_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
inputs = os.path.join(base_path, 'data','input')
outputs = os.path.join(base_path, 'data','output')

## Supplementary Code for Visualization

In [ ]:
def display_images(image):
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8, 6))
    plt.imshow(image_rgb)
    plt.axis('off')
    plt.show()

## Task 1 : Use image features to detect traffic signs in street images. **(7)**

a) Identify the type of noise and denoise the images. Use these denoised images for the next tasks. **(1)**

In [ ]:
def denoise(input_path, output_path):
    # Apply median filtering
    img = cv2.imread(input_path, cv2.IMREAD_COLOR)
    filtered_img = cv2.medianBlur(img, 3) # Use 3 x 3 kernel
    cv2.imwrite(output_path, filtered_img)
    return filtered_img

In [ ]:
image_path = os.path.join(inputs, 'phase1.jpg')
output_path = os.path.join(outputs, 'denoised_image.jpg')

image = denoise(image_path, output_path)
display_images(image)

b) Detect regions corresponding to traffic signs. **(3)**

In [ ]:
def segment_traffic_signs(image_path, output_path):
    # Using color based segmentation (blue)
    lower_blue = (100, 150, 50)
    upper_blue = (120, 255, 255)
    img = cv2.imread(image_path)
    #Change the color image to hsv channel and apply blue mask
    hsv_img = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    mask_blue = cv2.inRange(hsv_img, lower_blue, upper_blue)
    
    # Create contour to track traffic signs
    contours, _ = cv2.findContours(mask_blue, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    for cnt in contours:
        area =cv2.contourArea(cnt)
        
        # filter out smaller area
        if (area > 400):
            # Rectangle shapes detection (using Douglas-Peuker algorithm)
            peri = cv2.arcLength(cnt, True)
            approx = cv2.approxPolyDP(cnt, 0.02*peri, True)
            # filtering non rectangular shapes
            if len(approx) != 4:
                continue        
            x, y, w, h = cv2.boundingRect(approx)
            # The bounding box of the signs should have a reasonable ratio between width and height
            # This can avoid some blue segments in the sky or walls
            aspect_ratio = float(w) / h
            if aspect_ratio < 0.6 or aspect_ratio > 1.4:
                continue
            # draw contour in the original image
            img = cv2.drawContours(img, [cnt], -1, (0, 255, 0), 6)
            
    cv2.imwrite(output_path, img)
    return img

In [ ]:
image_path = os.path.join(outputs, 'denoised_image.jpg')
output_path = os.path.join(outputs, 'color_segmented_image.jpg')

image = segment_traffic_signs(image_path, output_path)
display_images(image)

c) Refine detected region boundaries with appropriate methods. **(3)**

In [ ]:
def refine_traffic_signs(image_path, output_path):

    img = cv2.imread(image_path)

    # Define the green color (from drawContours)
    lower_green = np.array([0, 250, 0])
    upper_green = np.array([0, 255, 0])

    # Mask green rectangles (in BGR)
    mask = cv2.inRange(img, lower_green, upper_green)

    # apply morphological cleaning to fill small gaps if any
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    refined = img.copy()
    cntrs, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cntrs = cntrs[0] if len(cntrs) == 2 else cntrs[1]
    for c in cntrs:
        area = cv2.contourArea(c)
        if area < 500:
            continue

        # Get bounding box and extract ROI from original image
        x, y, w, h = cv2.boundingRect(c)
        roi = img[y : y + h, x : x + w]

        # Convert ROI to grayscale and blur
        gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        blurred = cv2.GaussianBlur(gray, (5, 5), 0)

        # Canny edge detection
        edges = cv2.Canny(blurred, 50, 150)

        # Find refined contours from edges
        edge_contours, _ = cv2.findContours(
            edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        for ec in edge_contours:
            # Offset the contour to match original image coordinates
            ec = ec + np.array([[[x, y]]])
            approx = cv2.approxPolyDP(ec, 0.01 * cv2.arcLength(ec, True), True)
            if len(approx) >= 4:  # optionally ensure valid polygon
                cv2.drawContours(refined, [approx], -1, (0, 255, 0), 2)

    # Save and return
    cv2.imwrite(output_path, refined)
    return refined

In [ ]:
image_path = os.path.join(outputs, 'color_segmented_image.jpg')
output_path = os.path.join(outputs, 'refined_traffic_signs.jpg')

image = refine_traffic_signs(image_path, output_path)
display_images(image)

## Task 2 : Feature extraction and detection of pedestrians. **(8)**

a) Use an appropriate alogorithm to detect pedestrians. Draw bounding boxes around the detected pedestrians. **(3)**

In [ ]:
# To select the best bounding box
def non_max_suppression(boxes, overlapThresh):
    # if the input empty return
    if len(boxes) == 0:
        return []

    boxes = boxes.astype("float")
    pick = []

    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]

    area = (x2 - x1 + 1) * (y2 - y1 + 1)
    idx = np.argsort(y2)

    while len(idx) > 0:
        last = len(idx) - 1
        i = idx[last]
        pick.append(i)
        suppress = [last]

        for pos in range(0, last):
            j = idx[pos]

            xx1 = max(x1[i], x1[j])
            yy1 = max(y1[i], y1[j])
            xx2 = min(x2[i], x2[j])
            yy2 = min(y2[i], y2[j])

            w = max(0, xx2 - xx1 + 1)
            h = max(0, yy2 - yy1 + 1)

            overlap = float(w * h) / area[j]

            if overlap > overlapThresh:
                suppress.append(pos)

        idx = np.delete(idx, suppress)

    return boxes[pick].astype("int")

    
    

In [ ]:
def detect_pedestrians(image_path, output_path):
    # Using HoG and SVM 
    hog = cv2.HOGDescriptor()
    hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

    img = cv2.imread(image_path)
    rects, weights = hog.detectMultiScale(img, winStride=(8, 8), padding=(8, 8), scale=1.1)

    # filter out very small boxes before non-suppression
    rects = [r for r in rects if r[2] > 50 and r[3] > 100]

    r = np.array([[x, y, x + w, y + h] for x, y, w, h in rects])
    pick = non_max_suppression(r, overlapThresh=0.65)

    # draw rectangles in yellow
    for xa, ya, xb, yb in pick:
        cv2.rectangle(img, (xa, ya), (xb, yb), (0, 255, 255), 4)

    cv2.imwrite(output_path, img)
    return img


In [ ]:
input_path = os.path.join(outputs, 'denoised_image.jpg')
output_path = os.path.join(outputs, 'pedestrians.jpg')

image = detect_pedestrians(input_path, output_path)
display_images(image)

b) Detect faces of pedestrian and draw a bounding box around detected faces. **(3)**

In [ ]:
def detect_faces(input_path, output_path):
    # Using Cascade Classifier
    img = cv2.imread(input_path)
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Load the pre-trained Haar Cascade classifier for face detection
    face_cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + "haarcascade_frontalface_alt.xml"
    )

    faces = face_cascade.detectMultiScale(
        gray_img, scaleFactor=1.1, minNeighbors=6, minSize=(10, 10)
    )

    for x, y, w, h in faces:
        cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 255), 3)
    cv2.imwrite(output_path, img)
    return img

In [ ]:
input_path = os.path.join(outputs, 'denoised_image.jpg')
output_path = os.path.join(outputs, 'faces.jpg')

image = detect_faces(input_path, output_path)
display_images(image)

c) Briefly discuss the methods used for the above tasks. **(2)**

1. 
2. In the detect_faces task, we use the Haar Cascades method.

`haarcascade_frontalface_alt.xml` is an XML file containing a serialized Haar cascade face detector from the OpenCV library.

This algorithm is based on Viola-Jones Object Detection Framework. Basically, it is a coded list of decision trees, where each node tests one Haar feature, and each path ends with a decision that this is a face or not. The AdaBoost Classifier is applied here to select features for the next cascading step. Then, the cascading step implements Multiple stages of increasingly complex classifiers.  This method also combines with the sliding window technique on a pyramid scale. A fixed-size window slides through the image and uses the Haar Cascade classifier to detect faces.

We use "alt" instead of "default" because it was trained on another dataset and provides better results in this case.
